# 🧩 Sudoku AI — Entraînement GPU (Colab / Kaggle)
**EMSI — Module Deep Learning 2025-2026**

Ce notebook entraîne les **6 architectures DL** sur GPU free :
MLP · CNN · RNN · LSTM · GRU · Hybrid CNN+LSTM

| Étape | Description |
|-------|-------------|
| 1 | GPU + dépendances |
| 2 | Dataset (Kaggle auto ou génération) |
| 3 | Définition des architectures |
| 4 | Entraînement + early stopping |
| 5 | Évaluation + courbes |
| 6 | Téléchargement des poids `.pt` |

**Runtime recommandé** : GPU T4 (Colab gratuit) ou P100 (Kaggle gratuit)

## ⚙️ Étape 1 — GPU & Dépendances

In [1]:
import torch, sys
print(f'Python      : {sys.version.split()[0]}')
print(f'PyTorch     : {torch.__version__}')
print(f'GPU         : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU name    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM        : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('⚠ Pas de GPU — allez dans Runtime → Changer le type d\'exécution → GPU T4')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nDevice      : {DEVICE}')

Python      : 3.12.5
PyTorch     : 2.10.0+cpu
GPU         : False
⚠ Pas de GPU — allez dans Runtime → Changer le type d'exécution → GPU T4

Device      : cpu


In [2]:
import os, json, random, time, shutil
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
plt.style.use('dark_background')

# Dossiers — compatible Colab + Kaggle
if os.path.exists('/content'):
    WORK_DIR = Path('/content/sudoku_ai')
elif os.path.exists('/kaggle/working'):
    WORK_DIR = Path('/kaggle/working/sudoku_ai')
else:
    WORK_DIR = Path('.')

WEIGHTS_DIR = WORK_DIR / 'weights'
DATA_DIR    = WORK_DIR / 'data'
for d in [WORK_DIR, WEIGHTS_DIR, DATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# GPU setup
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_GPUS = torch.cuda.device_count()
AMP_OK = torch.cuda.is_available()

# Reproductibilité
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

print(f'✓ Imports OK')
print(f'  Device  = {DEVICE}  ({N_GPUS} GPU{"s" if N_GPUS>1 else ""})')
print(f'  AMP     = {AMP_OK}  (Mixed Precision FP16)')
print(f'  WORK    = {WORK_DIR}')


## 📦 Étape 2 — Dataset

**Option A (recommandée)** — Kaggle 1M Sudoku (~113 MB)  
Mettez `USE_KAGGLE = True` et configurez vos identifiants Kaggle une fois.

**Option B** — Génération par backtracking (aucun téléchargement, ~5 min)  
Laissez `USE_KAGGLE = False`.

In [3]:
# ══════════════════════════════════════════════════════════
# CHOIX DU DATASET — modifiez ici
# ══════════════════════════════════════════════════════════
USE_KAGGLE = True   # ← True pour télécharger 1M puzzles depuis Kaggle

# Tailles du dataset (adaptez selon votre runtime)
#  T4×2 Kaggle: TRAIN_SIZE = 1_000_000  (~30 min / modèle)
#  P100 Kaggle: TRAIN_SIZE = 1_000_000
#  Test rapide: TRAIN_SIZE = 50_000  (~5 min / modèle)
TRAIN_SIZE = 1_000_000
VAL_SIZE   =  20_000
TEST_SIZE  =  10_000
BATCH_SIZE = 2048  # AMP + T4×2 : 1024/GPU × 2 (FP16 = 2× moins de mémoire)
# ══════════════════════════════════════════════════════════

KAGGLE_CSV = None

if USE_KAGGLE:
    try:
        import subprocess
        subprocess.run(['pip', 'install', 'kagglehub', '-q'], check=False)
        import kagglehub
        from pathlib import Path
        print('Téléchargement Kaggle bryanpark/sudoku (~113 MB) via kagglehub...')
        path = kagglehub.dataset_download("bryanpark/sudoku")
        csv_candidate = Path(path) / 'sudoku.csv'
        if csv_candidate.exists():
            KAGGLE_CSV = str(csv_candidate)
            print(f'✓ Dataset Kaggle téléchargé : {KAGGLE_CSV}')
        else:
            print(f'✗ Fichier sudoku.csv introuvable dans {path}')
            print('  → Basculement en mode génération')
    except Exception as e:
        print(f'✗ Erreur kagglehub : {e}')
        print('  → Basculement en mode génération')

if KAGGLE_CSV:
    print(f'\nSource : Kaggle CSV ({KAGGLE_CSV})')
else:
    print('\nSource : génération par backtracking')



Source : génération par backtracking


In [4]:
# ══════════════════════════════════════════════════════════
# Utilitaires Sudoku
# ══════════════════════════════════════════════════════════

def _valid(board, r, c, n):
    if n in board[r]: return False
    if any(board[i][c] == n for i in range(9)): return False
    br, bc = 3*(r//3), 3*(c//3)
    return not any(board[br+dr][bc+dc] == n for dr in range(3) for dc in range(3))

def _solve(board, shuffle=True):
    for r in range(9):
        for c in range(9):
            if board[r][c] == 0:
                nums = list(range(1,10))
                if shuffle: random.shuffle(nums)
                for n in nums:
                    if _valid(board, r, c, n):
                        board[r][c] = n
                        if _solve(board, shuffle): return True
                        board[r][c] = 0
                return False
    return True

def gen_puzzle(n_clues=30):
    sol = [[0]*9 for _ in range(9)]
    _solve(sol)
    puz = [r[:] for r in sol]
    cells = list(range(81)); random.shuffle(cells)
    for idx in cells[:81-n_clues]:
        puz[idx//9][idx%9] = 0
    p = np.array([v for row in puz for v in row], dtype=np.int64)
    s = np.array([v for row in sol for v in row], dtype=np.int64)
    return p, s

def augment(puz, sol, factor=3):
    """Augmentation : rotations D4 + permutations bandes + remapping chiffres."""
    results = [(puz.copy(), sol.copy())]
    ops = [np.rot90, np.flipud, np.fliplr]
    for _ in range(factor - 1):
        p, s = puz.reshape(9,9).copy(), sol.reshape(9,9).copy()
        op = random.choice(ops)
        p, s = op(p).flatten(), op(s).flatten()
        # Permutation de bandes
        bp = random.sample(range(3), 3)
        p = np.vstack([p.reshape(9,9)[b*3:(b+1)*3] for b in bp]).flatten()
        s = np.vstack([s.reshape(9,9)[b*3:(b+1)*3] for b in bp]).flatten()
        # Remapping chiffres
        m = list(range(1,10)); random.shuffle(m)
        pm, sm = p.copy(), s.copy()
        for old, new in enumerate(m, 1):
            pm[p==old] = new; sm[s==old] = new
        results.append((pm, sm))
    return results

print('✓ Utilitaires Sudoku chargés')

✓ Utilitaires Sudoku chargés


In [5]:
# ══════════════════════════════════════════════════════════
# Chargement / Génération du dataset
# ══════════════════════════════════════════════════════════

def load_split(name, size, csv_path=None, aug_factor=3):
    cache = DATA_DIR / f'{name}.npz'
    if cache.exists():
        d = np.load(cache)
        p, s = d['puzzles'][:size], d['solutions'][:size]
        print(f'  {name:5s}: cache → {len(p):,} samples')
        return p, s

    if csv_path and Path(csv_path).exists():
        import pandas as pd
        print(f'  {name:5s}: chargement CSV ({size:,} lignes)...')
        df = pd.read_csv(csv_path, nrows=size)
        qc = next(c for c in df.columns if 'quiz' in c.lower() or 'puzzle' in c.lower())
        sc = next(c for c in df.columns if 'sol' in c.lower())
        def row2arr(s):
            return np.array([int(c) if c != '.' else 0 for c in str(s).strip()], dtype=np.int64)
        puzzles   = np.stack(df[qc].map(row2arr).values)
        solutions = np.stack(df[sc].map(row2arr).values)
    else:
        base = max(size // aug_factor, 1000)
        print(f'  {name:5s}: génération {base:,} puzzles (×{aug_factor} aug)...')
        puz_list, sol_list = [], []
        for _ in tqdm(range(base), desc=name, leave=False):
            nc = random.randint(25, 36)
            p, s = gen_puzzle(nc)
            puz_list.append(p); sol_list.append(s)
            if aug_factor > 1 and name == 'train':
                for ap, as_ in augment(p, s, aug_factor)[1:]:
                    puz_list.append(ap); sol_list.append(as_)
        idx = np.random.permutation(len(puz_list))
        puzzles   = np.stack(puz_list)[idx][:size]
        solutions = np.stack(sol_list)[idx][:size]

    np.savez_compressed(cache, puzzles=puzzles, solutions=solutions)
    print(f'  {name:5s}: {len(puzzles):,} samples  → {cache}')
    return puzzles, solutions


print('Préparation du dataset...')
train_p, train_s = load_split('train', TRAIN_SIZE, KAGGLE_CSV, aug_factor=3)
val_p,   val_s   = load_split('val',   VAL_SIZE,   KAGGLE_CSV, aug_factor=1)
test_p,  test_s  = load_split('test',  TEST_SIZE,  KAGGLE_CSV, aug_factor=1)

print(f'\n✓ Dataset prêt')
print(f'  Train : {len(train_p):>9,}')
print(f'  Val   : {len(val_p):>9,}')
print(f'  Test  : {len(test_p):>9,}')

clues = (train_p != 0).sum(1)
print(f'  Indices/puzzle : {clues.mean():.1f} ± {clues.std():.1f} [{clues.min()}-{clues.max()}]')

Préparation du dataset...
  train: génération 166,666 puzzles (×3 aug)...


train:   0%|          | 0/166666 [00:00<?, ?it/s]

  train: 499,998 samples  → data\train.npz
  val  : génération 10,000 puzzles (×1 aug)...


  val  : 10,000 samples  → data\val.npz
  test : génération 5,000 puzzles (×1 aug)...


  test : 5,000 samples  → data\test.npz

✓ Dataset prêt
  Train :   499,998
  Val   :    10,000
  Test  :     5,000
  Indices/puzzle : 30.5 ± 3.5 [25-36]


In [6]:
class SudokuDataset(Dataset):
    def __init__(self, puzzles, solutions):
        self.P = torch.tensor(puzzles,   dtype=torch.long)
        self.S = torch.tensor(solutions, dtype=torch.long)
    def __len__(self): return len(self.P)
    def __getitem__(self, i): return self.P[i], self.S[i]

def make_loader(p, s, shuffle):
    return DataLoader(
        SudokuDataset(p, s),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=2,
        pin_memory=AMP_OK,
        persistent_workers=True,
    )

train_loader = make_loader(train_p, train_s, shuffle=True)
val_loader   = make_loader(val_p,   val_s,   shuffle=False)
test_loader  = make_loader(test_p,  test_s,  shuffle=False)

print(f'✓ DataLoaders — {len(train_loader):,} batches/époque  (batch={BATCH_SIZE})')


✓ DataLoaders — 489 batches train (batch=1024)


## 🧠 Étape 3 — Architectures Deep Learning

In [7]:
# ══════════════════════════════════════════════════════════
# 1. MLP — Perceptron Multicouche
#    Entrée : one-hot (B, 810) → Sortie : (B, 81, 9)
# ══════════════════════════════════════════════════════════
class MLPSolver(nn.Module):
    name = 'MLP'
    def __init__(self, hidden=[1024,512,256,128], dropout=0.3):
        super().__init__()
        layers, d = [], 810
        for h in hidden:
            layers += [nn.Linear(d,h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            d = h
        layers.append(nn.Linear(d, 729))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        x = F.one_hot(x, 10).float().view(x.size(0), -1)
        return self.net(x).view(-1, 81, 9)

# ══════════════════════════════════════════════════════════
# 2. CNN — Réseau Convolutif
#    Grille 9×9 traitée comme image (10 canaux one-hot)
# ══════════════════════════════════════════════════════════
class CNNSolver(nn.Module):
    name = 'CNN'
    def __init__(self, ch=64, blocks=8, dropout=0.2):
        super().__init__()
        layers = [nn.Conv2d(10,ch,3,padding=1), nn.BatchNorm2d(ch), nn.ReLU()]
        for _ in range(blocks-1):
            layers += [nn.Conv2d(ch,ch,3,padding=1), nn.BatchNorm2d(ch), nn.ReLU()]
        layers += [nn.Dropout2d(dropout), nn.Conv2d(ch,9,1)]
        self.cnn = nn.Sequential(*layers)
    def forward(self, x):
        g = F.one_hot(x,10).float().view(-1,9,9,10).permute(0,3,1,2)
        return self.cnn(g).permute(0,2,3,1).contiguous().view(-1,81,9)

# ══════════════════════════════════════════════════════════
# 3. RNN — Réseau Récurrent Simple (BiRNN)
#    Séquence de 81 cellules, bidirectionnel
# ══════════════════════════════════════════════════════════
class RNNSolver(nn.Module):
    name = 'RNN'
    def __init__(self, emb=16, hidden=256, layers=2, dropout=0.3):
        super().__init__()
        self.emb = nn.Embedding(10, emb)
        self.rnn = nn.RNN(emb, hidden, layers, batch_first=True,
                          dropout=dropout if layers>1 else 0, bidirectional=True)
        self.fc = nn.Linear(hidden*2, 9)
    def forward(self, x):
        out, _ = self.rnn(self.emb(x))
        return self.fc(out)

# ══════════════════════════════════════════════════════════
# 4. LSTM — Long Short-Term Memory (BiLSTM)
#    3 portes : entrée, oubli, sortie + LayerNorm
# ══════════════════════════════════════════════════════════
class LSTMSolver(nn.Module):
    name = 'LSTM'
    def __init__(self, emb=32, hidden=256, layers=2, dropout=0.3):
        super().__init__()
        self.emb  = nn.Embedding(10, emb)
        self.lstm = nn.LSTM(emb, hidden, layers, batch_first=True,
                            dropout=dropout if layers>1 else 0, bidirectional=True)
        self.ln   = nn.LayerNorm(hidden*2)
        self.fc   = nn.Linear(hidden*2, 9)
    def forward(self, x):
        out, _ = self.lstm(self.emb(x))
        return self.fc(self.ln(out))

# ══════════════════════════════════════════════════════════
# 5. GRU — Gated Recurrent Unit (BiGRU)
#    2 portes : mise à jour + reset (plus léger que LSTM)
# ══════════════════════════════════════════════════════════
class GRUSolver(nn.Module):
    name = 'GRU'
    def __init__(self, emb=32, hidden=256, layers=2, dropout=0.3):
        super().__init__()
        self.emb = nn.Embedding(10, emb)
        self.gru = nn.GRU(emb, hidden, layers, batch_first=True,
                          dropout=dropout if layers>1 else 0, bidirectional=True)
        self.ln  = nn.LayerNorm(hidden*2)
        self.fc  = nn.Linear(hidden*2, 9)
    def forward(self, x):
        out, _ = self.gru(self.emb(x))
        return self.fc(self.ln(out))

# ══════════════════════════════════════════════════════════
# 6. Hybrid — CNN spatial + LSTM temporel
#    CNN extrait les features 9×9, LSTM traite ligne par ligne
# ══════════════════════════════════════════════════════════
class HybridSolver(nn.Module):
    name = 'Hybrid'
    def __init__(self, ch=64, hidden=256, layers=2, dropout=0.3):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(10,ch,3,padding=1), nn.BatchNorm2d(ch), nn.ReLU(),
            nn.Conv2d(ch,ch*2,3,padding=1), nn.BatchNorm2d(ch*2), nn.ReLU(),
            nn.Conv2d(ch*2,ch,3,padding=1), nn.BatchNorm2d(ch), nn.ReLU(),
        )
        self.lstm = nn.LSTM(ch*9, hidden, layers, batch_first=True,
                            bidirectional=True, dropout=dropout if layers>1 else 0)
        self.ln   = nn.LayerNorm(hidden*2)
        self.fc   = nn.Linear(hidden*2, 81)
    def forward(self, x):
        B = x.size(0)
        g = F.one_hot(x,10).float().view(B,9,9,10).permute(0,3,1,2)
        c = self.cnn(g).permute(0,2,1,3).contiguous().view(B,9,-1)
        out, _ = self.lstm(c)
        return self.fc(self.ln(out)).view(B,9,9,9).view(B,81,9)

# ── Vérification des shapes ──────────────────────────────
x_test = torch.zeros(4, 81, dtype=torch.long)
print(f'  {"Modèle":<10} {"Paramètres":>12} {"Shape sortie"}')
print(f'  {"-"*40}')
for cls in [MLPSolver, CNNSolver, RNNSolver, LSTMSolver, GRUSolver, HybridSolver]:
    m = cls()
    out = m(x_test)
    assert out.shape == (4,81,9), f'{cls.name}: {out.shape}'
    params = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'  {cls.name:<10} {params/1e6:>9.2f}M   {tuple(out.shape)}')
print('\n✓ Toutes les architectures validées')

  Modèle       Paramètres Shape sortie
  ----------------------------------------
  MLP             1.62M   (4, 81, 9)
  CNN             0.27M   (4, 81, 9)


  RNN             0.54M   (4, 81, 9)
  LSTM            2.18M   (4, 81, 9)
  GRU             1.63M   (4, 81, 9)
  Hybrid          3.48M   (4, 81, 9)

✓ Toutes les architectures validées


## 🏋️ Étape 4 — Entraînement

In [ ]:
# ══════════════════════════════════════════════════════════
# Entraînement complet
#   ✓ AMP torch.amp (API PyTorch 2.x)
#   ✓ logits.float() en validation → évite NaN FP16
#   ✓ AdamW + CosineWarmRestarts + LabelSmoothing
#   ✓ DataParallel automatique
#   ✓ Checkpoint toutes les 10 époques
# ══════════════════════════════════════════════════════════

def compute_metrics(logits, solutions, puzzles):
    preds   = logits.argmax(-1) + 1
    correct = (preds == solutions).float()
    empty   = (puzzles == 0)
    return {
        'cell_acc':   correct.mean().item(),
        'puzzle_acc': correct.all(-1).float().mean().item(),
        'empty_acc':  (correct * empty).sum().item() / empty.sum().clamp(min=1).item(),
    }


def train_model(model, tr_loader, va_loader,
                epochs=300, lr=1e-3, wd=1e-4, patience=50):

    model_name = model.name
    model      = model.to(DEVICE)
    ckpt_path  = WEIGHTS_DIR / f'{model_name.lower()}_ckpt.pt'

    if N_GPUS > 1:
        model = nn.DataParallel(model)
        print(f'  DataParallel : {N_GPUS} GPUs')

    crit   = nn.CrossEntropyLoss(label_smoothing=0.05)
    opt    = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched  = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=20, T_mult=2, eta_min=1e-6)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP_OK)

    best_val, best_state, no_imp = float('inf'), None, 0
    best_val_cell_acc, best_epoch = 0.0, 1
    H = {'train_loss':[], 'val_loss':[], 'train_cell_acc':[],
         'val_cell_acc':[], 'val_puzzle_acc':[], 'val_empty_cell_acc':[]}
    total_start = time.time()
    start_epoch = 1

    # ── Reprise depuis checkpoint ─────────────────────────
    if ckpt_path.exists():
        print(f'  Checkpoint trouvé — reprise...')
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        base = model.module if isinstance(model, nn.DataParallel) else model
        base.load_state_dict(ckpt['model_state'])
        opt.load_state_dict(ckpt['opt_state'])
        sched.load_state_dict(ckpt['sched_state'])
        scaler.load_state_dict(ckpt['scaler_state'])
        best_val          = ckpt['best_val']
        best_state        = ckpt['best_state']
        best_val_cell_acc = ckpt['best_val_cell_acc']
        best_epoch        = ckpt['best_epoch']
        no_imp            = ckpt['no_imp']
        H                 = ckpt['history']
        start_epoch       = ckpt['epoch'] + 1
        total_start      -= ckpt['elapsed_sec']
        print(f'  ✓ Reprise époque {start_epoch}  (best_val={best_val:.4f})')

    for epoch in range(start_epoch, epochs + 1):

        # ── Train ─────────────────────────────────────────
        model.train()
        tl = tca = 0
        bar = tqdm(tr_loader, desc=f'[{model_name}] Ep {epoch}/{epochs}', leave=False)
        for puz, sol in bar:
            puz, sol = puz.to(DEVICE, non_blocking=True), sol.to(DEVICE, non_blocking=True)
            opt.zero_grad()
            with torch.amp.autocast('cuda', enabled=AMP_OK):
                logits = model(puz)
                loss   = crit(logits.view(-1, 9), (sol - 1).view(-1))
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            tl  += loss.item()
            tca += (logits.argmax(-1) + 1 == sol).float().mean().item()
            bar.set_postfix(loss=f'{loss.item():.4f}')

        tl  /= len(tr_loader)
        tca /= len(tr_loader)

        # ── Validation ────────────────────────────────────
        model.eval()
        vl = vca = vpa = vea = 0
        with torch.no_grad():
            for puz, sol in va_loader:
                puz, sol = puz.to(DEVICE, non_blocking=True), sol.to(DEVICE, non_blocking=True)
                with torch.amp.autocast('cuda', enabled=AMP_OK):
                    logits = model(puz)
                # cast FP32 → évite overflow NaN en FP16
                logits = logits.float()
                vl  += crit(logits.view(-1, 9), (sol - 1).view(-1)).item()
                m    = compute_metrics(logits, sol, puz)
                vca += m['cell_acc'];  vpa += m['puzzle_acc'];  vea += m['empty_acc']

        vl  /= len(va_loader);  vca /= len(va_loader)
        vpa /= len(va_loader);  vea /= len(va_loader)
        sched.step(epoch)

        H['train_loss'].append(tl);       H['val_loss'].append(vl)
        H['train_cell_acc'].append(tca);  H['val_cell_acc'].append(vca)
        H['val_puzzle_acc'].append(vpa);  H['val_empty_cell_acc'].append(vea)

        lr_now = opt.param_groups[0]['lr']
        print(f'[{model_name}] Ep {epoch:3d}/{epochs} '
              f'Loss {tl:.4f}/{vl:.4f}  '
              f'Cell {tca:.3f}/{vca:.3f}  '
              f'Puzzle {vpa:.3f}  Vides {vea:.3f}  '
              f'LR {lr_now:.2e}')

        if vl < best_val:
            best_val = vl;  best_val_cell_acc = vca;  best_epoch = epoch;  no_imp = 0
            base = model.module if isinstance(model, nn.DataParallel) else model
            best_state = {k: v.cpu().clone() for k, v in base.state_dict().items()}
        else:
            no_imp += 1
            if no_imp >= patience:
                print(f"  Early stopping à l'époque {epoch}")
                break

        # ── Checkpoint toutes les 10 époques ──────────────
        if epoch % 10 == 0:
            base = model.module if isinstance(model, nn.DataParallel) else model
            torch.save({
                'epoch':             epoch,
                'model_state':       {k: v.cpu().clone() for k, v in base.state_dict().items()},
                'opt_state':         opt.state_dict(),
                'sched_state':       sched.state_dict(),
                'scaler_state':      scaler.state_dict(),
                'best_val':          best_val,
                'best_state':        best_state,
                'best_val_cell_acc': best_val_cell_acc,
                'best_epoch':        best_epoch,
                'no_imp':            no_imp,
                'history':           H,
                'elapsed_sec':       time.time() - total_start,
            }, ckpt_path)

    # ── Sauvegarde finale ─────────────────────────────────
    base = model.module if isinstance(model, nn.DataParallel) else model
    base.load_state_dict(best_state)
    wp = WEIGHTS_DIR / f'{model_name.lower()}.pt'
    torch.save(best_state, wp)
    if ckpt_path.exists(): ckpt_path.unlink()

    hist = {
        **H,
        'epochs_trained':      len(H['train_loss']),
        'best_val_loss':       best_val,
        'best_val_puzzle_acc': max(H['val_puzzle_acc']),
        'best_val_cell_acc':   best_val_cell_acc,
        'best_epoch':          best_epoch,
        'train_time_sec':      round(time.time() - total_start, 1),
    }
    with open(WEIGHTS_DIR / f'{model_name.lower()}_history.json', 'w') as fh:
        json.dump(hist, fh, indent=2)

    print(f'✓ {model_name} → {wp}  ({(time.time()-total_start)/60:.1f} min)')
    return base, hist


print(f'✓ train_model prêt  [AMP={AMP_OK} | AdamW | Checkpoint | NaN-safe]')


In [ ]:

# ══════════════════════════════════════════════════════════
# LANCER L'ENTRAÎNEMENT
# ══════════════════════════════════════════════════════════
EPOCHS   = 100   # 100 époques
PATIENCE = 40    # patience adaptée (T_0=20, cycle2=40ep)

# Architectures agrandies pour meilleure précision
MODELS_TO_TRAIN = [
    # MLP : 5 couches cachées, plus large
    MLPSolver(hidden=[2048, 1024, 512, 256, 128], dropout=0.2),
    # CNN : 128 canaux, 12 blocs résiduels
    CNNSolver(ch=128, blocks=12, dropout=0.1),
    # RNN : embedding 32, hidden 512, 3 couches
    RNNSolver(emb=32, hidden=512, layers=3, dropout=0.3),
    # LSTM : embedding 64, hidden 512, 3 couches
    LSTMSolver(emb=64, hidden=512, layers=3, dropout=0.2),
    # GRU : embedding 64, hidden 512, 3 couches
    GRUSolver(emb=64,  hidden=512, layers=3, dropout=0.2),
    # Hybrid : CNN 128 canaux + LSTM 512, 3 couches
    HybridSolver(ch=128, hidden=512, layers=3, dropout=0.2),
]

all_models    = {}
all_histories = {}

for model in MODELS_TO_TRAIN:
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'\n{"="*60}')
    print(f'  Modèle : {model.name}  ({params/1e6:.2f}M params)')
    print(f'{"="*60}')
    trained, hist = train_model(
        model,
        tr_loader=train_loader,
        va_loader=val_loader,
        epochs=EPOCHS,
        patience=PATIENCE,
    )
    all_models[model.name]    = trained
    all_histories[model.name] = hist
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print('\n✅ Tous les modèles entraînés !')



  Modèle : MLP


[MLP] Ep   1/30 Loss 2.1197/2.0114  Cell 0.203/0.300  Puzzle 0.000


[MLP] Ep   2/30 Loss 2.0625/1.9909  Cell 0.239/0.312  Puzzle 0.000


[MLP] Ep   3/30 Loss 2.0561/1.9852  Cell 0.243/0.315  Puzzle 0.000


[MLP] Ep   4/30 Loss 2.0540/1.9829  Cell 0.243/0.314  Puzzle 0.000


...

[RNN] Ep  10/30 Loss 0.8922/0.8840  Cell 0.607/0.611  Puzzle 0.000


[RNN] Ep  11/30 Loss 0.8847/0.8792  Cell 0.610/0.612  Puzzle 0.000


[RNN] Ep  12/30 Loss 0.8787/0.8697  Cell 0.612/0.616  Puzzle 0.000


[RNN] Ep  13/30 Loss 0.8722/0.8640  Cell 0.615/0.619  Puzzle 0.000


[RNN] Ep 14/30:  69%|██████▊   | 336/489 [34:06<13:03,  5.12s/it, loss=0.8569]

## 📊 Étape 5 — Évaluation & Visualisation

In [ ]:

# Évaluation sur le test set
results = {}
for name, model in all_models.items():
    model.eval()
    acc = {'cell_acc': 0, 'puzzle_acc': 0, 'empty_acc': 0}
    with torch.no_grad():
        for puz, sol in tqdm(test_loader, desc=f'Test {name}', leave=False):
            puz, sol = puz.to(DEVICE), sol.to(DEVICE)
            for k, v in compute_metrics(model(puz), sol, puz).items():
                acc[k] += v
    n = len(test_loader)
    results[name] = {k: v / n for k, v in acc.items()}

print(f'\n{"="*68}')
print(f'  {"Modèle":<10} {"CellAcc":>10} {"PuzzleAcc":>12} {"EmptyAcc":>12}')
print(f'  {"-"*56}')
best = max(results, key=lambda n: results[n]['puzzle_acc'])
for name, m in results.items():
    tag = ' ← meilleur' if name == best else ''
    print(f'  {name:<10} {m["cell_acc"]*100:>9.2f}%'
          f' {m["puzzle_acc"]*100:>11.2f}%'
          f' {m["empty_acc"]*100:>11.2f}%{tag}')
print(f'{"="*68}')


In [ ]:
# Courbes d'apprentissage
COLORS = {'MLP':'#6366f1','CNN':'#8b5cf6','RNN':'#ec4899',
          'LSTM':'#f59e0b','GRU':'#10b981','Hybrid':'#3b82f6'}

fig, axes = plt.subplots(2, 3, figsize=(18,10))
for i, (name, H) in enumerate(all_histories.items()):
    ax  = axes[i//3][i%3]
    ax2 = ax.twinx()
    col = COLORS.get(name,'#fff')
    ep  = range(1, len(H['train_loss'])+1)
    ax.plot(ep, H['train_loss'], color=col,      lw=2,   label='Train Loss')
    ax.plot(ep, H['val_loss'],   color='#ef4444', lw=2, ls='--', label='Val Loss')
    ax2.plot(ep, [v*100 for v in H['val_cell_acc']],        color='#10b981', lw=1.5, label='Cell Acc %')
    ax2.plot(ep, [v*100 for v in H['val_puzzle_acc']],      color='#f59e0b', lw=1.5, label='Puzzle Acc %')
    if H.get('val_empty_cell_acc'):
        ax2.plot(ep, [v*100 for v in H['val_empty_cell_acc']], color='#a78bfa', lw=1.2, ls=':', label='Vides Acc %')
    best_pa = max(H['val_puzzle_acc'])*100
    best_ep = H.get('best_epoch', '?')
    t_sec   = H.get('train_time_sec', '?')
    ax.set_title(f'{name}  —  Best PuzzleAcc: {best_pa:.1f}%  (ep#{best_ep}, {t_sec}s)', color='white', fontsize=10)
    ax.set_facecolor('#12122a')
    ax.tick_params(colors='white'); ax2.tick_params(colors='white')
    lines = ax.get_lines() + ax2.get_lines()
    ax.legend(lines, [l.get_label() for l in lines], fontsize=7, loc='upper right')

fig.patch.set_facecolor('#0a0a1a')
plt.suptitle("Courbes d'apprentissage — Sudoku AI", color='white', fontsize=15)
plt.tight_layout()
plt.savefig(WORK_DIR/'learning_curves.png', dpi=120, bbox_inches='tight', facecolor='#0a0a1a')
plt.show()
print('✓ Sauvegardé : learning_curves.png')


In [ ]:
# Graphique comparatif
names = list(results.keys()); x = np.arange(len(names)); w = 0.26
fig, ax = plt.subplots(figsize=(13,6))
ax.bar(x-w, [results[n]['cell_acc']*100   for n in names], w, label='Cell Acc',   color='#6366f1', alpha=.85)
ax.bar(x,   [results[n]['puzzle_acc']*100 for n in names], w, label='Puzzle Acc', color='#f59e0b', alpha=.85)
ax.bar(x+w, [results[n]['empty_acc']*100  for n in names], w, label='Empty Acc',  color='#10b981', alpha=.85)
for xi, name in enumerate(names):
    ax.text(xi, results[name]['puzzle_acc']*100+1,
            f'{results[name]["puzzle_acc"]*100:.1f}%',
            ha='center', color='#f59e0b', fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(names, color='white', fontsize=12)
ax.set_ylabel('Score (%)', color='white'); ax.set_ylim(0, 115)
ax.set_title('Comparaison finale — Test Set', color='white', fontsize=14)
ax.legend(fontsize=11); ax.set_facecolor('#12122a'); ax.tick_params(colors='white')
ax.grid(axis='y', alpha=.2)
fig.patch.set_facecolor('#0a0a1a')
plt.tight_layout()
plt.savefig(WORK_DIR/'model_comparison.png', dpi=120, bbox_inches='tight', facecolor='#0a0a1a')
plt.show()
print('✓ Sauvegardé : model_comparison.png')

## 💾 Étape 6 — Téléchargement des poids

In [ ]:
# Créer une archive ZIP de tous les poids + historiques
zip_path = WORK_DIR / 'weights_all'
shutil.make_archive(str(zip_path), 'zip', WEIGHTS_DIR)
zip_file = Path(str(zip_path) + '.zip')
print(f'✓ Archive : {zip_file}  ({zip_file.stat().st_size/1e6:.1f} MB)')

print('\nFichiers dans l\'archive :')
for f in sorted(WEIGHTS_DIR.iterdir()):
    print(f'  {f.name:<35} {f.stat().st_size/1e6:.2f} MB')

# Téléchargement
try:
    from google.colab import files
    files.download(str(zip_file))
    print('\n✓ Téléchargement lancé (Colab)')
except ImportError:
    # Kaggle : copier dans /kaggle/working/
    kaggle_out = Path('/kaggle/working/weights_all.zip')
    shutil.copy(zip_file, kaggle_out)
    print(f'\n✓ Fichier disponible dans Output → {kaggle_out}')

In [ ]:
print("""
╔══════════════════════════════════════════════════════════╗
║   Utiliser les poids sur votre machine locale            ║
╠══════════════════════════════════════════════════════════╣
║  1. Dézipper  weights_all.zip                            ║
║  2. Copier le contenu dans :                             ║
║       e:/sudoko_ai/backend/weights/                      ║
║  3. Lancer l'API (sans réentraîner) :                    ║
║       python run.py --skip-train                         ║
║  4. Ouvrir : http://localhost:3001                        ║
║                                                          ║
║  Les 6 modèles seront chargés automatiquement.           ║
╚══════════════════════════════════════════════════════════╝
""")